# Branching sandbox

Interactive walkthrough of the Git-style branching API exposed by `istari_experimental`.

You will:

1. Connect to a system.
2. List its branches.
3. Create a feature branch off `main` (or the system baseline).
4. Stage resource and subsystem changes, then commit.
5. Inspect the resulting tracked files / subsystems.
6. Merge the feature branch back into `main`.
7. Optionally archive the feature branch.

> **Heads up** — every step talks to the live platform. Use a sandbox / dev system; commits and merges *do* move tag pointers.

## 1. Connect to the platform

Reads `ISTARI_PAT` and `ISTARI_ENVIRONMENT_URL` from `.env` (or environment).

In [1]:
from istari_experimental import IstariPlatform

platform = IstariPlatform.from_env()
platform

IstariPlatform(url='?')

## 2. Pick a system

Set `SYSTEM_NAME` below to the system you want to experiment on. The cell after it lists the system's existing branches.

In [2]:
SYSTEM_NAME = "QA"  # <-- change to your sandbox system, OR set SYSTEM_ID below.
SYSTEM_ID = ""  # if you know the id, paste it here for an instant lookup.

if SYSTEM_ID:
    system = platform.get_system(SYSTEM_ID, by_id=True, verbose=True)
else:
    # `verbose=True` prints per-page progress so you can see it's not hung.
    # On large tenants `list_systems` can be slow; consider setting SYSTEM_ID above.
    system = platform.get_system(SYSTEM_NAME, verbose=True)

print(system)
print("baseline snapshot:", system.baseline.id)
print("baseline config :", system.baseline.configuration.name)

[get_system] fetching page 1 (size=100) ...


/Users/raphael/GitHub/istari-digital/istari-digital-client-cookbook/experimental/.venv/lib/python3.12/site-packages/istari_digital_client/log_utils.py:32: UserWarning: SDK is incompatible with Istari Registry v0.0.1+dev.commit.b13eba7 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
  result = func(*args, **kwargs)
2026-04-29 15:52:59 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v0.0.1+dev.commit.b13eba7 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.


[get_system] matched on page 1 after 7 systems
System('QA', id=ad11303d-1eda-4951-8db2-c367dd6bc991)
baseline snapshot: a4f50971-8390-4801-b556-8c5c4f63cf07
baseline config : v0


## 3. List existing branches

Each `BranchView` wraps a `SnapshotTag` and exposes the snapshot it currently points at.

In [3]:
branches = system.list_branches()
print(f"{len(branches)} branches on {system.name!r}:")
for b in branches:
    flag = " (baseline)" if b.is_baseline else ""
    print(f"  - {b.name:30s}  snapshot={b.snapshot_id}{flag}")

3 branches on 'QA':
  - sandbox/feature-20260428-233453  snapshot=9963fbe0-a763-485d-b2b2-14a53a14ad82
  - sandbox/feature-20260428-231036  snapshot=85ad32b6-3361-439c-a889-0c5dde012fa3
  - baseline                        snapshot=a4f50971-8390-4801-b556-8c5c4f63cf07 (baseline)


## 3b. List the resources (revisions) of the baseline branch

Fetch the `"baseline"` branch with `system.get_branch("baseline")` and print its tracked files together with the underlying `FileRevision` each one points at.

In [5]:
baseline_branch = system.get_branch("sandbox/feature-20260428-231036")
print(f"Baseline branch: {baseline_branch.name!r}  snapshot={baseline_branch.snapshot_id}")
print(f"Working configuration: {baseline_branch.configuration.name}\n")

tracked = baseline_branch.get_resources()
print(f"{len(tracked)} tracked file(s):")
for tf in tracked:
    rev_id = tf.pinned_file_revision_id or tf.current_file_revision_id
    rev_name = "?"
    try:
        rev = platform.client.get_revision(rev_id) if rev_id else None
        if rev is not None:
            rev_name = rev.display_name or rev.name or rev_id
    except Exception as e:  # noqa: BLE001 -- best-effort lookup
        rev_name = f"<error: {e.__class__.__name__}>"
    status = (tf.archive_status or "").lower()
    flag = "" if status == "active" else f"  [{tf.archive_status}]"
    print(f"  - {rev_name:40s}  resource_id={tf.resource_id}  rev={rev_id}  tf_status={tf.archive_status}{flag}")

Baseline branch: 'sandbox/feature-20260428-231036'  snapshot=85ad32b6-3361-439c-a889-0c5dde012fa3
Working configuration: branch:sandbox/feature-20260428-231036

1 tracked file(s):
  - Group3-UAS-Requirements.xlsx              resource_id=c11be13c-028c-4bdc-8a3b-d8cadfe388fc  rev=a906ce70-03cc-401c-b313-0d14065ce6b1  tf_status=Active


## 4. Create a feature branch

`SystemView.create_branch(name, *, description=None, from_branch=None, resources=None, subsystems=None)` materialises a brand-new `SnapshotTag` pointing at a fresh `SystemConfiguration` named `branch:<name>`.

The platform refuses to create *empty* configurations, so a branch always needs at least one tracked file. Seeding rules:

1. **Explicit seeds** — pass `resources=[...]` and/or `subsystems=[...]`.
   - `resources` items can be **paths on disk** (uploaded via `client.add_model` so the entry shows up as a *Resource*), pre-existing **Model id** strings (looked up first; falls back to raw `file_id` on miss), or any of `Model` / `File` / `FileRevision` / `TrackedFile` objects.
   - `subsystems` items can be a `BranchView`, an explicit `(system_id, tag_id)` tuple, or a `NewTrackedSystem`.
2. **`from_branch="..."`** — fork that branch's active tracked items (combined with any explicit seeds).
3. **Auto-README fallback** — if neither (1) nor (2) is provided, a small `README.md` is generated, uploaded as a Model, and tracked. Use `description=` to control its body.

There is **no implicit baseline fork** — forking is always opt-in via `from_branch=`. That means **`system.create_branch("my-feature")` works out of the box** without pre-staging anything. Three flavours below.

In [4]:
from datetime import datetime

FEATURE_NAME = f"sandbox/feature-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Simplest form: just a name (and an optional human-readable description).
# An auto-generated README.md is uploaded as a Model and tracked, so the
# branch is materialised correctly without any pre-staging.
feature = system.create_branch(
    FEATURE_NAME,
    description="Sandbox feature branch experimenting with the fluent API.",
)
print(feature)
print("  config  :", feature.configuration.name)
print("  snapshot:", feature.snapshot_id)
print("  tracked :", [tf.file_id for tf in feature.get_resources()])

Branch('sandbox/feature-20260429-155416', snapshot=9ee9586e-8ccd-4941-b663-b2b4a3e2e667)
  config  : branch:sandbox/feature-20260429-155416
  snapshot: 9ee9586e-8ccd-4941-b663-b2b4a3e2e667
  tracked : ['d31fb179-fb87-4ea8-916e-81fc08dbe8eb']


## 5. Inspect the branch's current state

`get_resources()` lists `TrackedFile`s; `get_subsystems()` lists `Subsystem`s. Both hit the API live.

In [5]:
resources = feature.get_resources()
subsystems = feature.get_subsystems()

print(f"resources  ({len(resources)}):")
for tf in resources[:10]:
    print(f"  - resource_id={tf.resource_id}  file_id={tf.file_id}  rev={tf.pinned_file_revision_id or tf.current_file_revision_id}")
if len(resources) > 10:
    print(f"  ... and {len(resources) - 10} more")

print(f"\nsubsystems ({len(subsystems)}):")
for s in subsystems:
    print(f"  - {s.name!r}  tag_id={s.tag_id}")

resources  (1):
  - resource_id=e79a10dd-edb6-4ec0-a7d3-da2b94861401  file_id=d31fb179-fb87-4ea8-916e-81fc08dbe8eb  rev=b7dd104c-96f9-4a34-870e-3e2c878df8b2

subsystems (0):


## 5a. Add a new Resource (file) to the branch

Upload a file from disk as a **Model** (so it shows up under the *Resources* tab in the UI), then stage it on the branch and commit.

The two-step pattern is:

1. `model = platform.client.add_model(path=..., display_name=...)` — upload as a `Model`.
2. `feature.add_resources(model.id).commit("...")` — stage the new resource on the branch and atomically materialise a new configuration + snapshot, advancing the branch tag.

`add_resources` tracks the model at **LATEST** (the branch follows future revisions). To pin the branch to a specific revision instead, use `add_revisions(...)` (see §5b).

In [6]:
from pathlib import Path

NEW_RESOURCE = Path("ELA Power Summary.txt")
assert NEW_RESOURCE.exists(), f"File not found: {NEW_RESOURCE.resolve()}"

ela_model = platform.client.add_model(
    path=str(NEW_RESOURCE),
    display_name=NEW_RESOURCE.stem,
    description="ELA power-summary handoff (sandbox demo).",
)
print(f"Uploaded Model id={ela_model.id}  file_id={ela_model.file.id}")

feature.add_resources(ela_model.id).commit(
    f"Add {NEW_RESOURCE.name} to {FEATURE_NAME}"
)

print(feature)
print("  snapshot:", feature.snapshot_id)
print("  tracked :")
for tf in feature.get_resources():
    print(f"    - resource_id={tf.resource_id}  file_id={tf.file_id}  rev={tf.pinned_file_revision_id or tf.current_file_revision_id}")

Uploaded Model id=8f360d7c-c265-4102-9faa-e63ca7833b5a  file_id=cb73cd6c-2863-498f-87e4-854dce2c4432
Branch('sandbox/feature-20260429-155416', snapshot=68613861-2f13-421e-b3a8-894922bc0f99)
  snapshot: 68613861-2f13-421e-b3a8-894922bc0f99
  tracked :
    - resource_id=8f360d7c-c265-4102-9faa-e63ca7833b5a  file_id=cb73cd6c-2863-498f-87e4-854dce2c4432  rev=150fd073-f280-43a6-8d28-116f74a9d973
    - resource_id=e79a10dd-edb6-4ec0-a7d3-da2b94861401  file_id=d31fb179-fb87-4ea8-916e-81fc08dbe8eb  rev=b7dd104c-96f9-4a34-870e-3e2c878df8b2


## 5b. Add a new revision of an existing Model to the branch

The cleanest way to make a tracked entry show up as a **Resource** in the UI is to bind it to a `Model` (a Resource is just a Model on the system). The flow is:

1. **Get the Resource** — fetch an existing `Model` from the platform (`platform.find_model(name=...)` or `platform.get_model(model_id)`).
2. **Upload a new Revision** to it — `model.upload_revision(path)` wraps `client.update_model(...)` and returns the new `FileRevision`.
3. **Pin that Revision on the branch** — `branch.add_revisions(rev).commit("...")` stages the revision LOCKED so the branch tracks that exact revision, not a moving "latest" pointer.

Set `MODEL_NAME` (or `MODEL_ID`) below and `NEW_REVISION_PATH` to the file you want to upload as the next revision of that model.

In [8]:
from pathlib import Path

MODEL_ID = ""               # paste a Model id for an instant fetch, OR leave empty
MODEL_NAME = "UAS-Requirements"  # display name to search for if MODEL_ID is empty
NEW_REVISION_PATH = Path("UAS-Requirements.xlsx")
COMMIT_MESSAGE = f"Pin updated {NEW_REVISION_PATH.name}"

assert NEW_REVISION_PATH.exists(), f"Not found: {NEW_REVISION_PATH.resolve()}"

# 1) Resolve the Model on the platform.
if MODEL_ID:
    model = platform.get_model(MODEL_ID)
else:
    model = platform.find_model(name=MODEL_NAME)
    if model is None:
        raise RuntimeError(
            f"No model named {MODEL_NAME!r} found. Set MODEL_ID directly or "
            f"use platform.upload_model(...) to create one first."
        )
print(f"Resource: {model}  (revisions so far: "
      f"{len(model.raw.file.revisions) if model.raw.file and model.raw.file.revisions else 0})")

# 2) Upload a new revision to that Model -- creates a new FileRevision under
#    the same File, and refreshes the ModelView in place.
new_rev = model.upload_revision(
    NEW_REVISION_PATH,
    display_name=NEW_REVISION_PATH.stem,
    version_name=f"sandbox-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
)
print(f"Uploaded revision: name={new_rev.name}  rev_id={new_rev.id}  file_id={new_rev.file_id}")

# 3) Stage the pinned revision on the feature branch and commit.
feature.add_revisions(new_rev).commit(COMMIT_MESSAGE)
print(f"Committed. Branch snapshot is now {feature.snapshot_id}")
print("Tracked files on the branch:")
for tf in feature.get_resources():
    pinned = tf.pinned_file_revision_id or "(LATEST)"
    print(f"  - resource_id={tf.resource_id}  file_id={tf.file_id}  pinned={pinned}  status={tf.archive_status}")

RuntimeError: No model named 'UAS-Requirements' found. Set MODEL_ID directly or use platform.upload_model(...) to create one first.

## 6. Stage changes and commit

Mutations are buffered locally — `has_pending_changes` flips to `True`. `commit()` then:

1. Builds the new tracked-file / tracked-subsystem list.
2. Creates a new `SystemConfiguration` (immutable on the platform).
3. Snapshots it.
4. Advances the branch tag pointer to that snapshot.

Set `RESOURCE_IDS_TO_ADD` to one or more `Model.id`s available on the platform, and `SUBSYSTEM_IDS_TO_ADD` to other `System.id`s. Leave the lists empty to skip that side.

In [ ]:
RESOURCE_IDS_TO_ADD: list[str] = []        # e.g. ["model-uuid-1", "model-uuid-2"]
RESOURCE_IDS_TO_REMOVE: list[str] = []     # e.g. ["model-uuid-old"]
SUBSYSTEM_IDS_TO_ADD: list[str] = []       # e.g. ["system-uuid-A"]
SUBSYSTEM_IDS_TO_REMOVE: list[str] = []

if not any([RESOURCE_IDS_TO_ADD, RESOURCE_IDS_TO_REMOVE, SUBSYSTEM_IDS_TO_ADD, SUBSYSTEM_IDS_TO_REMOVE]):
    print("Nothing staged -- fill in the lists above to exercise commit().")
else:
    pre_snapshot = feature.snapshot_id
    (feature
        .add_resources(*RESOURCE_IDS_TO_ADD)
        .remove_resources(*RESOURCE_IDS_TO_REMOVE)
        .add_subsystems(*SUBSYSTEM_IDS_TO_ADD)
        .remove_subsystems(*SUBSYSTEM_IDS_TO_REMOVE))
    print("pending changes:", feature.has_pending_changes)



In [8]:
feature.commit("sandbox: stage resource and subsystem changes")
print("committed.")
print(f"  snapshot moved: {pre_snapshot}  ->  {feature.snapshot_id}")
print(f"  has_pending_changes (after commit): {feature.has_pending_changes}")

committed.


NameError: name 'pre_snapshot' is not defined

## 7. Verify the new state

Re-read tracked files and subsystems from the live API after the commit.

In [ ]:
post_resources = feature.get_resources()
post_subsystems = feature.get_subsystems()

print(f"resources after commit  ({len(post_resources)}):")
for tf in post_resources[:10]:
    print(f"  - resource_id={tf.resource_id}  file_id={tf.file_id}")

print(f"\nsubsystems after commit ({len(post_subsystems)}):")
for s in post_subsystems:
    print(f"  - {s.name!r}  tag_id={s.tag_id}")

## 7b. Branch history — the tag's commit log

`BranchView.get_history()` is the equivalent of `git log <branch>`: it returns every snapshot the branch tag has *ever* pointed at, with timestamp, author, and snapshot id. Each entry is a `SnapshotTagRevision` from the SDK.

- Default order is **newest first** (matches `git log`); pass `newest_first=False` for chronological order.
- Archived revisions are filtered out by default; pass `include_archived=True` to see them.
- Use `feature.get_snapshot_at(rev)` to drill into the underlying `Snapshot`, then `client.get_configuration(snap.configuration_id)` and `client.list_tracked_files(configuration_id=cfg.id)` to inspect the working area as it was at that point.

The cell below walks the full log and, for each commit, prints the snapshot id, **the configuration name**, and **all tracked files** at that revision (best-effort resolving each tracked file to its revision display name). After §5a + §5b + §6 above, this branch should have **at least 4 entries**: the initial create, the resource add, the revision pin, and any §6 commits.

In [9]:
history = feature.get_history()

print(f"{feature.name!r}  --  {len(history)} commit(s) (newest first)\n")

# Cache config + tracked-file lookups so we make at most one round-trip per
# distinct snapshot/config across the loop.
_cfg_cache: dict[str, object] = {}
_files_cache: dict[str, list] = {}

def _resolve(rev):
    snap = feature.get_snapshot_at(rev)
    cfg = _cfg_cache.get(snap.configuration_id)
    if cfg is None:
        cfg = platform.client.get_configuration(snap.configuration_id)
        _cfg_cache[snap.configuration_id] = cfg
    files = _files_cache.get(cfg.id)
    if files is None:
        files = list(
            platform.client.list_tracked_files(configuration_id=cfg.id, size=100).iter_items()
        )
        _files_cache[cfg.id] = files
    return snap, cfg, files

for i, rev in enumerate(history):
    when = rev.created.strftime("%Y-%m-%d %H:%M:%S")
    marker = "HEAD ->" if i == 0 else "       "
    snap, cfg, files = _resolve(rev)
    print(f"{marker} {when}  by {rev.created_by_id}")
    print(f"         snapshot : {snap.id}")
    print(f"         config   : {cfg.name}  (id={cfg.id})")
    print(f"         files    : {len(files)}")
    for tf in files:
        rev_id = tf.pinned_file_revision_id or tf.current_file_revision_id
        flavour = "pinned" if tf.pinned_file_revision_id else "latest"
        # Best-effort: surface the revision's display name if cheap to fetch.
        rev_name = rev_id
        try:
            r = platform.client.get_revision(rev_id) if rev_id else None
            if r is not None:
                rev_name = r.display_name or r.name or rev_id
        except Exception as e:  # noqa: BLE001 -- best-effort lookup
            rev_name = f"<error: {e.__class__.__name__}>"
        print(f"           - {rev_name:40s}  resource_id={tf.resource_id}  rev={rev_id}  [{flavour}]")
    print()

'sandbox/feature-20260429-155416'  --  2 commit(s) (newest first):

  when                 author                                snapshot                              rev
  -------------------  ------------------------------------  ------------------------------------  ------------------------------------
  2026-04-29 19:58:25  80ddca7e-96c9-445a-b726-6f1455781879  68613861-2f13-421e-b3a8-894922bc0f99  cd485508-d1b5-4801-9ada-f17d2cbe4024
  2026-04-29 19:54:18  80ddca7e-96c9-445a-b726-6f1455781879  9ee9586e-8ccd-4941-b663-b2b4a3e2e667  3d8746c3-b616-4482-8ff0-60ac772bdfc5

Previous snapshot 9ee9586e-8ccd-4941-b663-b2b4a3e2e667 (config branch:sandbox/feature-20260429-155416):
  - resource_id=e79a10dd-edb6-4ec0-a7d3-da2b94861401  file_id=d31fb179-fb87-4ea8-916e-81fc08dbe8eb  rev=b7dd104c-96f9-4a34-870e-3e2c878df8b2


## 8. Chained mutations

Showcasing the fluent builder: every staged op returns `self`.

In [ ]:
SWAP_OUT: list[str] = []  # resource_ids currently tracked that you want to drop
SWAP_IN: list[str] = []   # new resource_ids to track instead

if not (SWAP_OUT or SWAP_IN):
    print("Nothing to swap -- fill SWAP_OUT / SWAP_IN to exercise chained mutations.")
else:
    pre = feature.snapshot_id
    (feature
        .remove_resources(*SWAP_OUT)
        .add_resources(*SWAP_IN)
        .commit("sandbox: swap resources"))
    print(f"  snapshot moved: {pre}  ->  {feature.snapshot_id}")

## 9. Merge the feature back into a target branch

`merge` is *ours-overwrite*: the target branch's tracked items are replaced wholesale with the source's, snapshotted, and the target tag is advanced.

Set `TARGET_BRANCH` to an existing branch on the system (e.g. `"main"`).

In [ ]:
TARGET_BRANCH = ""  # e.g. "main" -- leave empty to skip the merge step

if not TARGET_BRANCH:
    print("Skipping merge -- set TARGET_BRANCH to merge into.")
else:
    target_pre = system.get_branch(TARGET_BRANCH).snapshot_id
    merged = system.merge(
        from_branch=feature.name,
        to_branch=TARGET_BRANCH,
        message=f"Merge {feature.name} into {TARGET_BRANCH}",
    )
    print(f"merged into {merged.name!r}")
    print(f"  snapshot moved: {target_pre}  ->  {merged.snapshot_id}")

## 10. Cleanup (optional)

Archive the sandbox feature branch. Set `CONFIRM_ARCHIVE = True` to actually run it. `restore()` un-archives if you change your mind.

In [ ]:
CONFIRM_ARCHIVE = False

if not CONFIRM_ARCHIVE:
    print(f"Branch {feature.name!r} kept. Set CONFIRM_ARCHIVE = True to archive it.")
else:
    feature.archive()
    print(f"archived {feature.name!r}")
    # feature.restore()   # uncomment to un-archive

## 11. Re-list branches

Sanity check the final state.

In [ ]:
for b in system.list_branches():
    flag = " (baseline)" if b.is_baseline else ""
    print(f"  - {b.name:40s}  snapshot={b.snapshot_id}{flag}")